# Hypothesis Test: Do Songs by the Same Artist Fall in the Same Cluster?

**Python port of `Hypothesis_test.R`.**

**Hypothesis:** Songs with common artists fall in the same cluster.

**Null Hypothesis (H0):** There is no association between a song's artist and its cluster.

**Alternate Hypothesis (H1):** There is an association between a song's artist and its cluster.

We test this using a Pearson's Chi-squared test on a contingency table of artist vs. cluster.

**Change from the original**: the R version depended on a `clusters` variable already sitting in memory from having run `Recommender_model_2.R` in the same session first — fragile, since separate notebooks don't share variables even within the same language. This version recomputes the same k-means clustering itself, so it runs standalone.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.stats import chi2_contingency

## Load Data and Recompute Clustering

Same feature set and k=7 as `Recommender_model_2.ipynb`, so the clusters referenced here match.

In [2]:
DATA_DIR = 'data'
data = pd.read_csv(os.path.join(DATA_DIR, 'spotify-2023.csv'), encoding='latin-1')

FEATURE_COLS = [
    'bpm', 'danceability_%', 'valence_%', 'energy_%',
    'acousticness_%', 'instrumentalness_%', 'liveness_%', 'speechiness_%'
]

X_scaled = StandardScaler().fit_transform(data[FEATURE_COLS])

kmeans = KMeans(n_clusters=7, n_init=10, random_state=42)
data['cluster'] = kmeans.fit_predict(X_scaled)

## Build the Artist × Cluster Contingency Table

Songs can have multiple artists (comma-separated), so each song's cluster gets attributed to every one of its listed artists — `pandas`' `explode()` handles this cleanly, replacing the R original's manual loop-and-chunk workaround.

In [3]:
artist_cluster = data[['artist(s)_name', 'cluster']].copy()
artist_cluster['artist(s)_name'] = artist_cluster['artist(s)_name'].str.split(',')
artist_cluster = artist_cluster.explode('artist(s)_name').reset_index(drop=True)
artist_cluster['artist(s)_name'] = artist_cluster['artist(s)_name'].str.strip()

contingency_table = pd.crosstab(artist_cluster['artist(s)_name'], artist_cluster['cluster'])
print(f"Contingency table shape: {contingency_table.shape[0]} artists x {contingency_table.shape[1]} clusters")
contingency_table.head()

Contingency table shape: 699 artists x 7 clusters


cluster,0,1,2,3,4,5,6
artist(s)_name,,,,,,,
,0,1,0,0,0,0,0
(G)I-DLE,0,1,0,1,0,0,0
070 Shake,0,0,0,0,0,0,2
21 Savage,0,0,6,1,3,0,4
24kgoldn,0,1,0,0,0,0,0


## Chi-Square Test on a Random Sample of Artists

Matches the R original's approach of testing on a random subsample of artists (rather than the full table, which can have too many near-empty cells for a reliable chi-square test), repeated many times to check the result is stable rather than a fluke of one particular sample.

In [4]:
def sampled_chi_square_test(table, n_artists=100, random_state=None):
    rng = np.random.default_rng(random_state)
    sample_idx = rng.choice(table.index, size=min(n_artists, len(table)), replace=False)
    sub_table = table.loc[sample_idx]
    # drop clusters with all-zero counts in this sample, chi2_contingency requires nonzero row/col sums
    sub_table = sub_table.loc[:, (sub_table.sum(axis=0) > 0)]
    sub_table = sub_table.loc[(sub_table.sum(axis=1) > 0), :]
    chi2, p, dof, expected = chi2_contingency(sub_table)
    return chi2, p, dof

In [5]:
chi2, p, dof = sampled_chi_square_test(contingency_table, n_artists=100, random_state=42)
print(f"Chi-squared = {chi2:.2f}, df = {dof}, p-value = {p:.6g}")
# a very low p-value (< 0.05) => reject the null hypothesis
# i.e. songs by the same artist tend to fall in the same cluster

Chi-squared = 698.27, df = 594, p-value = 0.00196692


## Repeating the Test Many Times to Check Stability of the p-value

In [6]:
n_repeats = 1000
p_values = np.empty(n_repeats)

for i in range(n_repeats):
    _, p_i, _ = sampled_chi_square_test(contingency_table, n_artists=100, random_state=i)
    p_values[i] = p_i

print(f"Mean p-value across {n_repeats} random samples: {np.nanmean(p_values):.6g}")
print(f"Share of samples with p < 0.05: {np.mean(p_values < 0.05):.1%}")

Mean p-value across 1000 random samples: 0.023758
Share of samples with p < 0.05: 91.9%
